# GEPA Prompt Optimizer — Bot de soporte al cliente

**Caso de uso:** un bot de soporte recibe preguntas de usuarios junto con un fragmento de la base de conocimiento (contexto). Con el seed prompt `"Eres un asistente de soporte."`, el bot ignora el contexto y responde con información genérica o inventada. GEPA optimiza el prompt para que el bot use únicamente el contexto, sea conciso y no alucine.

El dataset contiene 4 temas de soporte (facturación, acceso, almacenamiento, permisos) con 2 preguntas cada uno. Cada `Dataset` tiene su propio `context` — el fragmento de FAQ relevante para esas preguntas.

**Sin evaluador custom.** GEPA usa el `LLMEvaluator` interno con el `objective` como criterio.

## Setup

In [1]:
import os
import logging
import getpass

logging.getLogger("httpx").setLevel(logging.WARNING)

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[3]))
sys.path.insert(0, str(Path().resolve().parents[3] / "examples"))

## 1. Retriever

Carga las preguntas de soporte y sus respuestas esperadas. Cada `Dataset` incluye el fragmento de FAQ relevante en `context` — el mismo que se inyecta al agente en cada consulta.

In [3]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset

class SupportRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        data_path = Path("../data/dataset.json")
        with open(data_path, encoding="utf-8") as f:
            return [Dataset.model_validate(item) for item in json.load(f)]

## 2. Model

El modelo que GEPA usa para **generar prompts candidatos mejorados**. No es el agente que se está optimizando — ese usa el mismo modelo con el prompt candidato como system prompt.

In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

## 3. Run the optimizer

- `seed_prompt`: el prompt actual del agente — deliberadamente vago.
- `objective`: describe en lenguaje natural qué hace un buen agente de soporte. GEPA lo usa para evaluar respuestas y guiar la generación de mejores prompts.
- Sin `evaluator` explícito: GEPA usa un juez LLM interno basado en el `objective`.

In [5]:
from fair_forge.prompt_optimizer.gepa import GEPAOptimizer

result = GEPAOptimizer.run(
    retriever=SupportRetriever,
    model=model,
    seed_prompt="Eres un asistente de soporte.",
    objective=(
        "Responder preguntas de soporte usando únicamente la información del contexto proporcionado. "
        "Las respuestas deben ser directas y concisas. "
        "No agregar información que no esté en el contexto. No inventar datos, precios, pasos ni funcionalidades."
    ),
    iterations=5,
    candidates_per_iteration=3,
)

Evaluating seed prompt on 8 examples...


Optimizing prompt:   0%|          | 0/5 [00:00<?, ?it/s]

## 4. Results

In [6]:
print(f"Initial score : {result.initial_score:.2f} / 1.00  (LLM judge · {result.n_examples} examples)")
print(f"Final score   : {result.final_score:.2f} / 1.00  (LLM judge · {result.n_examples} examples)")
print(f"Iterations run: {result.iterations_run}")
print()
print("=== Seed prompt ===")
print("Eres un asistente de soporte.")
print()
print("=== Optimized prompt ===")
print(result.optimized_prompt)

Initial score : 0.44 / 1.00  (LLM judge · 8 examples)
Final score   : 0.89 / 1.00  (LLM judge · 8 examples)
Iterations run: 1

=== Seed prompt ===
Eres un asistente de soporte.

=== Optimized prompt ===
Proporciona respuestas directas y concisas como asistente de soporte para Nexo, enfocándote en resolver el problema específico sin agregar detalles adicionales.
